# 🎯 AI Resume Analyzer - Complete Project Walkthrough

## Project Overview

- Build a production-ready resume screening system using LangChain
- PDF parsing and text extraction
- Structured output with Pydantic
- Multi-stage pipelines (LCEL syntax)
- Batch processing and ranking candidates

### Setup: imports and API key

## 📝 Section 1: Environment Setup

- Load API key from environment variables (.env file)
- Use python-dotenv for security
- Fail fast with ValueError if key is missing

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not set. Please add it to your .env or environment variables before running.")

### 2. Imports for LangChain & helpers

## 📝 Section 2: LangChain Imports

- ChatOpenAI for LLM calls
- PyPDFLoader to read PDFs
- PromptTemplate for reusable prompts
- PydanticOutputParser for structured output
- Pydantic BaseModel for data validation

In [14]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from pydantic import BaseModel, Field
import json


### 3. Load Resume PDF

## 📝 Section 3: Loading Resume PDFs

- Resumes folder should contain PDF files
- List comprehension filters .pdf files (case-insensitive)
- Store filename + docs together for tracking
- Handle multi-page resumes (each page is a Document object)

In [15]:
resume_folder = "resumes"

resume_files = [
    f for f in os.listdir(resume_folder)
    if f.lower().endswith(".pdf")
]

print("Found resumes:", resume_files)

all_docs = []

for file in resume_files:
    path = os.path.join(resume_folder, file)
    print(f"\nLoading: {file}")
    
    loader = PyPDFLoader(path)
    docs = loader.load()
    
    all_docs.append({
        "filename": file,
        "docs": docs
    })

print("\nCompleted loading all resumes.")

Found resumes: ['Alex Johnson.pdf', 'Arjun Nair.pdf']

Loading: Alex Johnson.pdf

Loading: Arjun Nair.pdf

Completed loading all resumes.


### Helper Function to Combine Pages & Clean Text

## 📝 Section 4: Text Cleaning

- Combine multi-page PDFs into single string
- Use regex to collapse whitespace
- Cleaner text = lower token cost
- Preserves paragraph structure with double newlines

In [16]:
import re

def combine_and_clean(docs):
    text = "\n\n".join([d.page_content for d in docs])
    text = re.sub(r"\s+", " ", text).strip()
    return text


### Initialize LLM

## 📝 Section 5: Initialize LLM

- Model: gpt-4o-mini (cost-effective but capable)
- temperature=0 for deterministic extraction
- API key authentication
- Reuse one LLM instance for all chains

In [17]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

### Pydantic Schema for structured JSON extraction

## 📝 Section 6: Pydantic Schema ⭐ CRITICAL

- EducationEntry: Nested schema for education history
- ResumeSchema: Main data structure for extracted resume
- Optional fields using `str | None`
- Use `Field(default_factory=list)` for mutable defaults
- Pydantic validates LLM output automatically

In [18]:
class EducationEntry(BaseModel):
    degree: str | None = None
    institution: str | None = None
    years: str | None = None
    cgpa: str | None = None

class ResumeSchema(BaseModel):
    name: str | None = None
    email: str | None = None
    phone: str | None = None
    skills: list[str] = Field(default_factory=list)
    experience_summary: str | None = None
    education: list[EducationEntry] = Field(default_factory=list)

In [19]:
# Example: Create a sample ResumeSchema object to demonstrate the structure
sample_resume = ResumeSchema(
    name="John Doe",
    email="john@example.com",
    phone="+1-555-0123",
    skills=["Python", "SQL", "Docker", "AWS"],
    experience_summary="5 years as a Data Engineer at Tech Corp",
    education=[
        EducationEntry(
            degree="Bachelor of Science",
            institution="State University",
            years="2018-2022",
            cgpa="3.8/4.0"
        )
    ]
)

print("📋 Sample ResumeSchema Object:")
print(sample_resume.model_dump_json(indent=2))

📋 Sample ResumeSchema Object:
{
  "name": "John Doe",
  "email": "john@example.com",
  "phone": "+1-555-0123",
  "skills": [
    "Python",
    "SQL",
    "Docker",
    "AWS"
  ],
  "experience_summary": "5 years as a Data Engineer at Tech Corp",
  "education": [
    {
      "degree": "Bachelor of Science",
      "institution": "State University",
      "years": "2018-2022",
      "cgpa": "3.8/4.0"
    }
  ]
}


In [20]:
parser = PydanticOutputParser(pydantic_object=ResumeSchema)

## 📝 Section 7: Extraction Chain ⭐ LCEL INTRODUCTION

- PromptTemplate with input variables
- LCEL pipe syntax: `prompt | llm`
- Temperature=0 ensures deterministic output
- Clean role definition for expert extraction

In [21]:
prompt_extract = PromptTemplate(
    input_variables=["resume_text"],
    template="""
You are an expert resume parser. Extract the following fields and return ONLY valid JSON:
name, email, phone, skills(list), experience_summary, education(list)

Resume:
{resume_text}
"""
)

# NEW: Replace LLMChain with prompt | llm pipeline
extract_chain = prompt_extract | llm


### Skill Gap Prompt + Pipeline (Updated)

## 📝 Section 8: Skill Gap Analysis Chain

- Inputs: candidate_skills, required_skills
- Outputs: missing_skills (list) + recommendations (dict)
- Same LCEL pipe pattern: `prompt | llm`
- Compares what candidate HAS vs what job NEEDS

In [22]:
skill_gap_prompt = PromptTemplate(
    input_variables=["candidate_skills", "required_skills"],
    template="""
Candidate skills: {candidate_skills}
Required skills: {required_skills}

Return ONLY JSON with:
- missing_skills (list)
- recommendations (object: skill -> recommendation)
"""
)

skill_gap_chain = skill_gap_prompt | llm


### Final Report Prompt + Pipeline

## 📝 Section 9: Final Report Chain

- Combines extracted resume + skill gap analysis
- Generates overall_score (0-100)
- Provides short_recommendation for HR
- Multi-step orchestration pattern

In [23]:
final_prompt = PromptTemplate(
    input_variables=["candidate_json", "skill_gap_json"],
    template="""
Generate a final candidate report with:
name, email, phone, skills, missing_skills, recommendations,
experience_summary, education, overall_score(0-100), short_recommendation.

Candidate JSON:
{candidate_json}

Skill Gap JSON:
{skill_gap_json}

Return ONLY JSON.
"""
)

final_chain = final_prompt | llm


### Job Description & Skill Extraction

## 📝 Section 10: Job Description Processing

- clean_json_text removes code fences from LLM output
- Extract required skills from job description
- Error handling with fallback skills list
- Graceful degradation pattern

In [24]:
def clean_json_text(text):
    """Remove code fences and clean JSON output from LLM responses."""
    if not isinstance(text, str):
        text = str(text)
    text = text.strip()
    # Remove ```json ... ``` fences
    fenced = re.search(r"```json\s*(.*?)```", text, re.DOTALL)
    if fenced:
        return fenced.group(1).strip()
    # Remove ``` ... ``` fences (any language)
    fenced_any = re.search(r"```\s*(.*?)```", text, re.DOTALL)
    if fenced_any:
        return fenced_any.group(1).strip()
    return text

In [25]:
# Job Description - Can be loaded from file or pasted here
JOB_DESCRIPTION = """
Senior Data Engineer
Requirements:
- Python (5+ years)
- SQL (advanced)
- PySpark or distributed computing
- AWS or GCP cloud platforms
- ETL pipeline design
- Pandas and NumPy
- Apache Airflow or similar orchestration
- Docker containerization
- Machine Learning fundamentals
- Git version control
Nice to Have:
- Kafka
- Scala
- TensorFlow/PyTorch
"""

# Extract required skills from JD using LLM
jd_extraction_prompt = PromptTemplate(
    input_variables=["job_description"],
    template="""
Extract ALL technical skills mentioned in this job description.
Return ONLY a JSON array of skill names as strings.

Job Description:
{job_description}

Return format: ["skill1", "skill2", ...]
"""
)

jd_extract_chain = jd_extraction_prompt | llm

jd_msg = jd_extract_chain.invoke({"job_description": JOB_DESCRIPTION})
jd_text = clean_json_text(jd_msg.content)

try:
    required_skills = json.loads(jd_text)
except Exception as e:
    print(f"\n⚠️ JD skill extraction failed: {e}")
    print(f"Raw output (preview): {jd_text[:500]}")
    required_skills = ["Python", "SQL", "AWS", "Pandas", "Machine Learning", "Docker"]

print(f"\n📊 Extracted Required Skills from JD: {required_skills}")


📊 Extracted Required Skills from JD: ['Python', 'SQL', 'PySpark', 'distributed computing', 'AWS', 'GCP', 'ETL pipeline design', 'Pandas', 'NumPy', 'Apache Airflow', 'orchestration', 'Docker', 'Machine Learning', 'Git', 'Kafka', 'Scala', 'TensorFlow', 'PyTorch']


### Pipeline Function (Modern Invocation)

## 📝 Section 11: The Main Pipeline Function ⭐ ORCHESTRATION

- Orchestrates extract → skill gap → final report
- Guard each step with try/except error handling
- Returns: (final_report, parsed_resume, skill_gap_json)
- Three-step chain composition pattern

In [26]:
def process_resume(resume_text: str):
    # --- 1) Extraction step ---
    extract_msg = extract_chain.invoke({"resume_text": resume_text[:4000]})
    extract_result = clean_json_text(extract_msg.content)

    try:
        parsed = parser.parse(extract_result)
    except Exception as e:
        print("\n⚠️ Extraction parsing failed:", e)
        print("Raw extraction (preview):", extract_result[:500])
        parsed = None

    # --- 2) Skill gap step ---
    candidate_skills = parsed.skills if parsed and parsed.skills else []

    sg_msg = skill_gap_chain.invoke({
        "candidate_skills": ", ".join(candidate_skills),
        "required_skills": ", ".join(required_skills)
    })
    sg_text = clean_json_text(sg_msg.content)

    try:
        skill_gap_json = json.loads(sg_text)
    except Exception:
        print("\n⚠️ Skill gap JSON parsing failed")
        print("Raw output (preview):", sg_text[:500])
        skill_gap_json = {}

    # --- 3) Final report step ---
    candidate_json_text = parsed.model_dump_json() if parsed else "{}"
    skill_gap_json_text = json.dumps(skill_gap_json) if skill_gap_json else sg_text

    final_msg = final_chain.invoke({
        "candidate_json": candidate_json_text,
        "skill_gap_json": skill_gap_json_text
    })
    final_result = clean_json_text(final_msg.content)

    try:
        final_report = json.loads(final_result)
    except Exception:
        print("\n⚠️ Final report JSON parsing failed")
        print("Raw (preview):", final_result[:500])
        final_report = {"raw": final_result}

    return final_report, parsed, skill_gap_json


### Process ALL Resumes + Save Outputs

## 📝 Section 12: Batch Processing

- Iterate through all loaded documents
- slugify filenames to create safe output names
- Write each result to outputs/<name>.json
- Track processed files to avoid duplicates

In [27]:
import os
import json
from pathlib import Path

os.makedirs("outputs", exist_ok=True)

def slugify_filename(name: str) -> str:
    base = Path(name).stem
    safe = "".join(ch if ch.isalnum() or ch in ("-", "_", " ") else "_" for ch in base).strip()
    return safe.replace(" ", "_") or "output"

processed_files = set()

for item in all_docs:
    filename = item["filename"]
    docs = item["docs"]

    base = slugify_filename(filename)
    if base in processed_files:
        print(f"Skipping duplicate entry for {base}")
        continue
    processed_files.add(base)

    # Combine and clean text
    text = combine_and_clean(docs)

    print(f"\n📄 Processing {filename}...")

    # Run the 3-step pipeline: Extraction → Skill Gap → Final Report
    final_report, parsed_resume, skill_gap = process_resume(text)

    # Save output JSON
    out_path = f"outputs/{base}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2)

    print(f"✅ Saved output to: {out_path}")


📄 Processing Alex Johnson.pdf...
✅ Saved output to: outputs/Alex_Johnson.json

📄 Processing Arjun Nair.pdf...
✅ Saved output to: outputs/Arjun_Nair.json


### View Results Summary

## 📝 Section 13: Ranking & Analysis ⭐ DATA SCIENCE

- Load all output JSON files
- Calculate Jaccard similarity (skill match %)
- Create ranking DataFrame
- Sort by similarity and LLM score

In [28]:
from difflib import SequenceMatcher
import pandas as pd

def calculate_similarity(candidate_skills: list, required_skills: list) -> float:
    """Calculate Jaccard similarity between candidate and required skills."""
    if not required_skills:
        return 0.0
    candidate_set = set(skill.lower() for skill in candidate_skills)
    required_set = set(skill.lower() for skill in required_skills)
    intersection = len(candidate_set & required_set)
    union = len(candidate_set | required_set)
    return round((intersection / union) * 100, 2) if union > 0 else 0.0

# Load all JSON outputs
output_files = [f for f in os.listdir("outputs") if f.endswith(".json")]

results = []
for file in output_files:
    with open(f"outputs/{file}", "r", encoding="utf-8") as f:
        data = json.load(f)
        candidate_skills = data.get("skills", [])
        similarity = calculate_similarity(candidate_skills, required_skills)
        results.append({
            "Name": data.get("name", "N/A"),
            "Email": data.get("email", "N/A"),
            "Phone": data.get("phone", "N/A"),
            "Skills": ", ".join(candidate_skills),
            "Score": data.get("overall_score", 0),
            "Similarity %": similarity,
            "Missing Skills": ", ".join(data.get("missing_skills", [])),
            "Recommendation": data.get("short_recommendation", "N/A")
        })

# Create DataFrame and sort by similarity score descending, then by overall_score
df = pd.DataFrame(results)
df = df.sort_values(by=["Similarity %", "Score"], ascending=False).reset_index(drop=True)
df.insert(0, "Rank", df.index + 1)

print("\n" + "="*120)
print("📊 CANDIDATE RANKING BASED ON JD MATCH")
print("="*120)
print(df[["Rank", "Name", "Score", "Similarity %", "Missing Skills"]].to_string(index=False))
print("="*120)


📊 CANDIDATE RANKING BASED ON JD MATCH
 Rank         Name  Score  Similarity %                                                                                                                                              Missing Skills
    1 Alex Johnson     75         27.27                       PySpark, distributed computing, AWS, ETL pipeline design, Pandas, NumPy, Apache Airflow, orchestration, Docker, Kafka, Scala, PyTorch
    2  Nisha Verma     75         27.27                       PySpark, distributed computing, AWS, ETL pipeline design, Pandas, NumPy, Apache Airflow, orchestration, Docker, Kafka, Scala, PyTorch
    3  Rahul Mehta     75         27.27                       PySpark, distributed computing, AWS, ETL pipeline design, Pandas, NumPy, Apache Airflow, orchestration, Docker, Kafka, Scala, PyTorch
    4     John Doe     75         27.27                       PySpark, distributed computing, AWS, ETL pipeline design, Pandas, NumPy, Apache Airflow, orchestration, Docker, Kaf

### Export to CSV

## 📝 Section 14: Export to CSV

- Save DataFrame to CSV for sharing
- Universal format (Excel, Google Sheets, etc.)
- index=False removes row numbers
- Ready for HR team distribution

In [29]:
csv_path = "outputs/candidate_ranking.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Candidate ranking saved to: {csv_path}")

✅ Candidate ranking saved to: outputs/candidate_ranking.csv


## 🎉 Project Complete!

**What We Built:**
- ✅ Production-grade resume screening system
- ✅ LCEL chains and multi-step pipelines
- ✅ Pydantic schema validation
- ✅ Batch processing and ranking
- ✅ Candidate scoring and skill gap analysis

**Key Learnings:**
- LangChain LCEL syntax for composing chains
- Structured output with Pydantic
- Error handling and graceful degradation
- ETL patterns (Extract → Transform → Load)
- Data analysis with Pandas

## 📚 Next Steps

**Beginner Extensions:**
- Add support for DOCX files
- Customize job description and prompts
- Extract additional fields (certifications, languages)

**Intermediate:**
- Build Streamlit UI for resume uploads
- Add data visualizations
- Implement weighted scoring

**Advanced:**
- Multi-language resume support
- Real-time webhook processing
- ATS system integration